# 1. FAIR Data & Metadata

Part I gave you the tools to clean and reshape a table once you already have
it in front of you. This notebook is about the step *before* that: making
sure the data is worth having in the first place — described well enough
that you (in six months) or a colleague (right now) can actually use it.

**Topics**
1. What metadata is, and why "the number without the label" is worthless
2. The FAIR principles — Findable, Accessible, Interoperable, Reusable
3. Representing metadata as Python dictionaries
4. JSON: saving and loading self-describing data
5. A minimal FAIR record for a real sample
6. Writing a FAIR checklist function

## 1.1 Data Without Metadata Is (Almost) Useless

Imagine you find a file called `results.csv` on an old hard drive with a
single column: `182, 195, 188, 201, 179, 193`. Without more information you
cannot answer a single scientific question about it. Is it hardness in HV?
A voltage in mV? Which sample? Which instrument? Measured by whom, when, and
under what conditions?

**Metadata** is simply *data about data* — the information that turns a bare
number into something you can trust and reuse. In a lab notebook this is the
context you write around a measurement almost without thinking about it:
sample ID, date, instrument, operator, units, calibration. The moment data
leaves a lab notebook and becomes a spreadsheet or a plot, that context is
exactly what tends to get lost — and that is the problem this notebook
solves.

In [ ]:
# ── The problem, made concrete ────────────────────────────────────────────────
bare_numbers = [182, 195, 188, 201, 179, 193]
print(bare_numbers)
print('What are these? Hardness? Capacity? Which sample? Which units?')
print('Without metadata, this list is scientifically worthless.')

## 1.2 The FAIR Principles

**FAIR** is a short checklist, first published by a group of scientists and
data stewards in 2016, for what "well-managed research data" means in
practice. It does **not** mean "open" or "free" — data can be FAIR and still
be access-controlled. The four letters:

| Principle | Plain-language question | Typical mechanism |
|---|---|---|
| **F**indable | Can someone (or some software) locate this dataset at all? | Persistent identifier (DOI), rich searchable metadata |
| **A**ccessible | Once found, can it actually be retrieved (by people or machines), even if under conditions? | Standard access protocol (HTTP/REST API), clear access/license terms |
| **I**nteroperable | Can it be combined with other datasets and tools without manual translation? | Shared vocabularies / ontologies (→ Notebook 2), standard formats (JSON, CSV, HDF5) |
| **R**eusable | Does it have enough context (provenance, licence, units) to be reused correctly? | Rich metadata, clear licence, documented methods |

Notice that three of the four letters are really about **metadata quality**,
not about the measured numbers themselves. This is why data management is a
skill worth learning explicitly rather than an afterthought: the difference
between a FAIR dataset and an unusable one is almost entirely in how it is
*described*, not how it was *measured*.

## 1.3 Metadata as a Python Dictionary

You already have the right tool for representing metadata: the `dict`
(Part I, Notebook 1). A dictionary maps a name to a value exactly the way
metadata maps a field ("operator", "instrument", "unit") to its content —
and unlike a spreadsheet, it can hold **nested** structure, which matters
because real metadata is rarely flat (an "instrument" might itself need a
name, a calibration date, and a serial number).

In [ ]:
# ── A single measurement, described properly ─────────────────────────────────
measurement = {
    'quantity':   'Vickers hardness',
    'value':      188,
    'unit':       'HV',
    'sample_id':  'STL-2024-014',
    'instrument': {
        'name':            'Struers Duramin-40',
        'serial_number':   'DM40-3381',
        'last_calibrated': '2024-11-02',
    },
    'operator':   'A. Lindqvist',
    'date':       '2025-01-15',
}

print('Quantity:', measurement['quantity'])
print('Value:   ', measurement['value'], measurement['unit'])
print('Instrument:', measurement['instrument']['name'])
print()
print('Compare to the bare number from Section 1.1 — this is now reusable.')

## 1.4 JSON: Saving Self-Describing Data

A Python dictionary only exists while your script is running. To make data
**Findable** and **Accessible** later, you need to write it to disk in a
format that preserves its structure — and ideally one that any programming
language, not just Python, can read back. **JSON** (JavaScript Object
Notation) is the standard choice: it maps almost one-to-one onto Python
dicts/lists/strings/numbers, is human-readable in a text editor, and is the
format NOMAD (Notebook 4) and most modern research-data infrastructure use
under the hood.

`json.dump` writes a Python object to a file; `json.load` reads it back —
the round trip is lossless for the data types JSON supports (numbers,
strings, booleans, `None`, lists, and nested dicts).

In [ ]:
import json

# ── Save the measurement as JSON ──────────────────────────────────────────────
with open('measurement.json', 'w', encoding='utf-8') as f:
    json.dump(measurement, f, indent=2)

# Inspect the raw text that was written
with open('measurement.json', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# ── Load it back and confirm the round trip is lossless ───────────────────────
with open('measurement.json', encoding='utf-8') as f:
    reloaded = json.load(f)

print(reloaded == measurement)
print(reloaded['instrument']['name'])

## 1.5 A Minimal FAIR Record for a Real Sample

Let's build a more complete example: metadata for a battery cathode
synthesis batch, structured so it satisfies each FAIR letter. Read through
the fields and match each one to F, A, I, or R in the table above before
running the cell.

In [ ]:
cathode_record = {
    # Findable
    'identifier':   'urn:uuid:8f14e45f-ceea-4b9f-8a5e-1a2b3c4d5e6f',   # a persistent, unique ID
    'title':        'LiFePO4 cathode, solid-state synthesis batch 07',
    'keywords':     ['LiFePO4', 'cathode', 'olivine', 'solid-state synthesis'],

    # Accessible
    'access':       'open',
    'licence':      'CC-BY-4.0',
    'contact':      'a.lindqvist@example.uu.se',

    # Interoperable
    'quantity_kind_iri': {                       # ties field names to a shared vocabulary (Notebook 2)
        'synthesis_temperature': 'emmo:ThermodynamicTemperature',
        'capacity':              'emmo:ElectricCharge',
    },
    'format':       'JSON',

    # Reusable
    'method':       'Solid-state reaction, 2-step calcination',
    'provenance':   'Synthesised from Li2CO3, FeC2O4·2H2O, NH4H2PO4 (stoichiometric ratio)',
    'created':      '2025-01-15',
    'creator':      'A. Lindqvist, Dept. of Chemistry, Uppsala University',

    # The actual data
    'parameters': {
        'synthesis_temperature_C': 750,
        'synthesis_time_h':        6,
        'atmosphere':              'Ar',
    },
    'results': {
        'capacity_mAh_g': 161.4,
        'coulombic_efficiency': 0.94,
    },
}

print(json.dumps(cathode_record, indent=2))

## 1.6 Writing a FAIR Checklist Function

A useful habit: write a small function that checks a record against the
FAIR principles *before* you save it, the same way you might sanity-check
data before running statistics on it. This will not catch everything FAIR
means in the full sense (that also involves how a *repository* handles your
data — see Notebook 4), but it catches the most common omissions in a
record you write yourself.

In [ ]:
def fair_check(record, required_fields=('identifier', 'title', 'licence',
                                       'creator', 'created', 'method')):
    """Return a dict of {field: present?} for a minimal FAIR self-check."""
    report = {field: (field in record and record[field] not in (None, '', []))
              for field in required_fields}
    return report


def print_fair_report(record, name='record'):
    report = fair_check(record)
    n_ok = sum(report.values())
    print(f'FAIR self-check for {name}: {n_ok}/{len(report)} fields present')
    for field, ok in report.items():
        flag = '✓' if ok else '✗ MISSING'
        print(f'  {flag}  {field}')


print_fair_report(cathode_record, 'cathode_record')
print()
print_fair_report(measurement, 'measurement')   # the Section 1.3 record — much sparser

---
## Exercises

1. **Fill the gaps**: `measurement` from Section 1.3 fails several checks in
   `fair_check`. Add the missing fields (`identifier`, `licence`, `method`,
   ...) so that it passes all six checks, then confirm with
   `print_fair_report`.

2. **Round trip with a list**: Create a Python list of three `measurement`-style
   dictionaries for three different samples. Save the whole list to
   `measurements.json` with `json.dump`, reload it, and verify with `==`
   that the reloaded object equals the original.

3. **Extend the checklist**: Add a check to `fair_check` for whether `keywords`
   contains at least 2 entries (searchability — Findable). Hint: you will
   need a special case in the dict comprehension, or a small helper function,
   since this check is not just "is the field present.